In [1]:
import os
import time
import json

from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape(path):

    options = ChromeOptions()
    # options.add_argument("--headless=new")  # comment out for visible Chrome
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    driver = Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 10)

    os.makedirs("html", exist_ok=True)

    database = {
        "auction": {},
        "vehicles": {}
    }

    driver.get(path)
    time.sleep(2)


    try:
        login_btn = wait.until(EC.element_to_be_clickable((By.XPATH, '//a[text()="Login"]')))
        login_btn.click()
        wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys("fourbrotherstrading@icloud.com")
        wait.until(EC.presence_of_element_located((By.ID, "password"))).send_keys("Muhssan7865#")
        wait.until(EC.element_to_be_clickable((By.XPATH, '//button[@type="submit"]'))).click()
        time.sleep(3)
        print("✅ Logged in")
    except Exception as e:
        print("Login skipped or failed:", e)


    try:
        database["auction"]["name"] = wait.until(
            EC.presence_of_element_located((By.XPATH, '//div[@class="listing-title"]/h2'))
        ).text.strip()

        database["auction"]["date"] = driver.find_element(
            By.XPATH, '//p[contains(text(),"Sale Date")]'
        ).text.replace("Sale Date:", "").strip()

        database["auction"]["time"] = driver.find_element(
            By.XPATH, '//p[contains(text(),"Sale Time")]'
        ).text.replace("Sale Time:", "").strip()

        print(f"📌 Auction: {database['auction']['name']} on {database['auction']['date']} at {database['auction']['time']}")
    except Exception as e:
        print("Auction info not found:", e)

    while True:
        wait.until(EC.presence_of_all_elements_located((By.XPATH, '//a[contains(@href,"/vehicle/")]')))
        car_links = driver.find_elements(By.XPATH, '//a[contains(@href,"/vehicle/")]')
        links = list({link.get_attribute("href") for link in car_links})

        main_window = driver.current_window_handle

        for link in links:

            driver.execute_script("window.open(arguments[0]);", link)
            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(1)

            try:
                wait.until(EC.presence_of_element_located((By.XPATH, '//th[text()="Registration Number"]')))
                reg = driver.find_element(
                    By.XPATH, '//th[text()="Registration Number"]/following-sibling::td'
                ).text.strip()

                if reg in database["vehicles"]:
                    print(f"⏭ Already saved {reg}")
                    driver.close()
                    driver.switch_to.window(main_window)
                    continue

                html_path = f"html/{reg}.html"
                with open(html_path, "w", encoding="utf-8") as f:
                    f.write(driver.page_source)

                database["vehicles"][reg] = {"html": html_path}
                print(f"✅ Saved {reg}")

            except Exception as e:
                print("Vehicle skipped:", e)


            driver.close()
            driver.switch_to.window(main_window)

        try:
            next_btn = driver.find_element(By.XPATH, '//li[contains(@class,"pag-next")]/a')
            next_url = next_btn.get_attribute("href")

            if not next_url:
                break


            if next_url.startswith("//"):
                next_url = "https:" + next_url
            elif next_url.startswith("/"):
                next_url = "https://www.cva-auctions.co.uk" + next_url

            print(f"➡ Moving to next page: {next_url}")
            driver.get(next_url)
            time.sleep(2)

        except Exception as e:
            print("No next page or pagination ended:", e)
            break


    with open("database.json", "w", encoding="utf-8") as f:
        json.dump(database, f, indent=4)

    driver.quit()
    print("✅ Scraping completed!")


scrape("https://www.cva-auctions.co.uk/auction/86")


✅ Logged in
📌 Auction: VAN & CAR AUCTION on 22nd January 2026 at 10:00 AM
✅ Saved HK18NWT
✅ Saved ST68UCO
✅ Saved PK21CKL
✅ Saved FJ07PXS
✅ Saved CE56XOM
✅ Saved PJ71NLM
✅ Saved SJ71HTT
✅ Saved LL69UZD
✅ Saved ST13NNW
✅ Saved YC13OJF
✅ Saved SA71KHW
✅ Saved ST62EJF
✅ Saved NU21OCZ
✅ Saved AK66JWJ
✅ Saved ST66VMR
✅ Saved FN62ACV
✅ Saved HJ16FNZ
✅ Saved LS10BUH
✅ Saved ST13NMY
✅ Saved SA71XCO
✅ Saved ST66UWB
✅ Saved SP11UXZ
✅ Saved EX66GVF
✅ Saved PF67WZY
✅ Saved GD70GGP
✅ Saved NA67KJN
✅ Saved SF65WOB
✅ Saved RV20UEP
✅ Saved SJ71HRW
✅ Saved PJ71NLA
➡ Moving to next page: https://www.cva-auctions.co.uk/auction/86?page=2
✅ Saved PK66ECC
✅ Saved SL23PYX
✅ Saved BL16YBJ
✅ Saved SD68YAV
✅ Saved YN06AZB
✅ Saved CP69BZS
✅ Saved PN71BVL
✅ Saved YT07ACZ
✅ Saved RE12PDV
✅ Saved OY69UCX
✅ Saved LK13CYA
✅ Saved SJ65YJW
✅ Saved SO71FWJ
✅ Saved PO17EPF
✅ Saved JO08ELC
✅ Saved AP72WAE
✅ Saved LX20NVT
✅ Saved KP69TNL
✅ Saved GJ70LUT
✅ Saved AE63HWZ
✅ Saved WX15FSS
✅ Saved MX56OBS
✅ Saved AO17NFJ
✅ Save

In [2]:
import os
import csv
import json
from bs4 import BeautifulSoup
from datetime import datetime
import re
with open(r"D:\bots\headers.json", "r", encoding="utf-8") as f:
    header_map = json.load(f)


headers = [header_map[k] for k in sorted(header_map, key=int)]

with open(r"database.json", "r", encoding="utf-8") as f:
    database = json.load(f)
auctionDetails=database.get("auction")
html_folder = os.path.join(os.getcwd(), "html")
csv_file = os.path.join(os.getcwd(), "cva_data.csv")

html_files = [f for f in os.listdir(html_folder) if f.endswith(".html")]
BASE_URL = "https://www.cva-auctions.co.uk"

def get_base_folder_info():
    folder_name = os.path.basename(os.getcwd())
    parts = folder_name.split("-")

    if not parts or not parts[0].isdigit():
        return "", ""

    sheet_id = parts[0]
    name_parts = parts[1:]

    if name_parts and name_parts[0].isdigit():
        name_parts = name_parts[1:]

    auction_name = "-".join(name_parts).strip()
    return sheet_id, auction_name

def get_table_value(soup, key):
    th = soup.find("th", string=lambda x: x and x.strip() == key)
    if not th:
        return ""

    td = th.find_next_sibling("td")
    if not td:
        return ""

    return td.get_text(strip=True)

def clean_mileage(mileage_str):
    if not mileage_str:
        return ""
    digits_only = "".join(c for c in mileage_str if c.isdigit())
    return digits_only

def sql_date(date_str):
    if not date_str:
        return ""
    clean = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date_str)

    return datetime.strptime(clean, "%d %B %Y").strftime("%Y-%m-%d")
def sql_year(date_str):
    if not date_str:
        return ""
    date_str = date_str.strip().replace("\\", "").strip()
    try:
        dt = datetime.strptime(date_str, "%d %b %Y")
        return str(dt.year)
    except:
        return ""

def cc_to_liters(cc_value):
    try:
        cc_int = int(cc_value)
        liters = cc_int / 1000
        return round(liters, 1) 
    except:
        return ""

def sql_date2(date_str):
    if not date_str:
        return ""
    date_str = date_str.strip().replace("\\", "").replace("/", "").strip()

    try:
        return datetime.strptime(date_str, "%d %b %Y").strftime("%Y-%m-%d")
    except:
        return ""
def sql_time(time_str):
    if not time_str:
        return ""

    return datetime.strptime(time_str, "%I:%M %p").strftime("%H:%M:%S")
def extract_vehicle_images(soup):
    images = []
    gallery = soup.find("ul", id="aos-primary-slider-list")
    if gallery:
        li_tags = gallery.find_all("li", class_="splide__slide")
        for li in li_tags:
            img_tag = li.find("img", src=True)
            if img_tag:
                src = img_tag['src'].strip()
                if src.startswith("//"):
                    src = "https:" + src
                elif src.startswith("/"):
                    src = BASE_URL + src
                images.append(src)
    
    all_images_csv = ",".join(images)
    damage_images_csv = ",".join(images[-4:]) if len(images) >= 4 else all_images_csv
    
    return all_images_csv, damage_images_csv

all_data = []

for html_file in html_files:
    path = os.path.join(html_folder, html_file)

    with open(path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")


    row = {h: "" for h in headers}

    sheet_id, auction_name = get_base_folder_info()
    a_tag = soup.find("a", class_=["text-primary", "d-flex", "align-items-center"], href=True)
    if a_tag:
        href = a_tag['href']
        if href.startswith("/"):
            full_url = "https://www.cva-auctions.co.uk" + href
        else:
            full_url = ""


    row[header_map["1"]] = auctionDetails.get("name", auction_name)
    row[header_map["2"]] = sheet_id
    row[header_map["3"]] = "Commercial Vehicle Auctions LTD"
    
    reg_no   = get_table_value(soup, "Registration Number")
    Dor   = get_table_value(soup, "Date of Registration")
    Make   = get_table_value(soup, "Make")
    Model   = get_table_value(soup, "Model")
    Variant   = get_table_value(soup, "Variant")
    Lot   = get_table_value(soup, "Lot Number")
    Bodytype   = get_table_value(soup, "Body Type")
    FuelType   = get_table_value(soup, "Fuel Type")
    Transmission   = get_table_value(soup, "Transmission")
    Mileage   = get_table_value(soup, "Mileage")
    MileageWarranted   = get_table_value(soup, "Mileage Warranted")
    Center   = get_table_value(soup, "Location")
    V5   = get_table_value(soup, "V5")
    MOT_Expiry   = get_table_value(soup, "MOT Expiry")
    ServiceHistory   = get_table_value(soup, "Service History")
    VATStatus   = get_table_value(soup, "VAT Status")
    Vendor   = get_table_value(soup, "Vendor")
    FormerKeepers   = get_table_value(soup, "Former Keepers")
    CC   = get_table_value(soup, "CC")
    Remarks   = get_table_value(soup, "Remarks")
    row[header_map["4"]] = f"{Make} {Model} {Variant}"
    row[header_map["5"]] = reg_no
    row[header_map["6"]] = Make
    row[header_map["7"]] = Model
    row[header_map["8"]] = Variant
    row[header_map["9"]] = Lot
    row[header_map["10"]] = Bodytype
    row[header_map["11"]] = FuelType
    row[header_map["12"]] = Transmission
    row[header_map["13"]] = Center
    row[header_map["14"]] = sql_date(auctionDetails.get("date", ""))
    row[header_map["15"]] = sql_time(auctionDetails.get("time", ""))
    row[header_map["16"]] = sql_date2(Dor)
    row[header_map["17"]] = sql_year(Dor)
    row[header_map["18"]] = clean_mileage(Mileage)
    row[header_map["19"]] = MileageWarranted
    row[header_map["20"]] = V5
    row[header_map["21"]] = MOT_Expiry
    row[header_map["22"]] = ServiceHistory
    row[header_map["23"]] = VATStatus
    row[header_map["24"]] = Vendor
    row[header_map["25"]] = FormerKeepers
    row[header_map["26"]] = cc_to_liters(CC)
    row[header_map["27"]] = Remarks
    row[header_map["28"]] = full_url
    
    all_images_str, damage_images_str = extract_vehicle_images(soup)
    row[header_map["29"]] = all_images_str
    row[header_map["30"]] = damage_images_str
    
    Length   = get_table_value(soup, "Length")
    Width   = get_table_value(soup, "Width")
    Height   = get_table_value(soup, "Height")
    Imported   = get_table_value(soup, "Imported")
    
    additional_info = {
        "Additional Information": {
            "Length": Length,
            "Width": Width,
            "Height": Height,
            "Imported": Imported
        }
    }
    row[header_map["32"]] = additional_info
    


    all_data.append(row)

with open(csv_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(all_data)

print(f"✅ CSV created successfully: {csv_file}")


✅ CSV created successfully: d:\bots\21-cva\cva_data.csv


In [3]:
import threading,requests,os
import pandas as pd
from urllib.parse import urlparse, urljoin
from PIL import Image, ImageDraw, ImageFont


def add_watermark_to_image(image_path, text="Commercial Vehicle Auctions LTD"):
    try:
        image = Image.open(image_path).convert("RGBA")
        txt_layer = Image.new("RGBA", image.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(txt_layer)

        font_size = max(20, image.width // 50) 
        try:
            font = ImageFont.truetype("arial.ttf", font_size)
        except:
            font = ImageFont.load_default()

        margin = int(font_size * 0.6)
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]
        x = image.width - text_width - margin
        y = image.height - text_height - margin


        box_padding = int(font_size * 0.4)
        draw.rectangle(
            [x - box_padding, y - box_padding, x + text_width + box_padding, y + text_height + box_padding],
            fill=(0, 0, 0, 180)
        )


        draw.text((x, y), text, font=font, fill=(255, 255, 255, 240))


        watermarked = Image.alpha_composite(image, txt_layer).convert("RGB")
        watermarked.save(image_path)
        print(f"✅ Watermark added to {image_path}")

    except Exception as e:
        print(f"❌ Failed to watermark {image_path}: {e}")



def download_images(data, main_folder="Images", column_name="Images"):
    os.makedirs(main_folder, exist_ok=True)

    for _, row in data.iterrows():
        reg_no = str(row["Reg"]).strip().replace("/", "_").replace("\\", "_")
        image_urls = row.get(column_name)

        if not image_urls or pd.isna(image_urls):
            print(f"⚠️ Skipping {reg_no}: no {column_name.lower()}.")
            continue

        reg_folder = os.path.join(main_folder, reg_no)
        os.makedirs(reg_folder, exist_ok=True)

        urls = [u.strip() for u in image_urls.split(",") if u.strip()]
        for idx, url in enumerate(urls, start=1):
            if "eu.cdn.autosonshow.tv/" in url:
                url = url.split("key=")[-1].split("&")[0]

            if not url.startswith(("http://", "https://")):
                url = "https://" + url.lstrip("/")

            parsed = urlparse(url)
            if not parsed.scheme or not parsed.netloc:
                print(f"⚠️ Invalid URL skipped: {url}")
                continue

            try:
                response = requests.get(url, stream=True, timeout=15)
                response.raise_for_status()

                ext = os.path.splitext(parsed.path)[1] or ".jpg"
                save_path = os.path.join(reg_folder, f"{idx}{ext}")

                with open(save_path, "wb") as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)

                add_watermark_to_image(save_path)
                print(f"✅ Saved: {save_path}")

            except Exception as e:
                print(f"❌ Failed to download {url} for {reg_no}: {e}")



def download_reports(data, main_folder="Inspection Report"):
    os.makedirs(main_folder, exist_ok=True)

    for _, row in data.iterrows():
        reg_no = str(row["Reg"]).strip().replace("/", "_").replace("\\", "_")
        report_url = row.get("Inspection Report")

        if not report_url or pd.isna(report_url):
            print(f"⚠️ Missing Inspection Report for {reg_no}")
            continue

        if not report_url.startswith(("http://", "https://")):
            report_url = "https://" + report_url.lstrip("/")

        reg_folder = os.path.join(main_folder, reg_no)
        os.makedirs(reg_folder, exist_ok=True)
        save_path = os.path.join(reg_folder, f"{reg_no}.pdf")

        try:
            response = requests.get(report_url, stream=True, timeout=15)
            response.raise_for_status()
            with open(save_path, "wb") as f:
                for chunk in response.iter_content(1024):
                    f.write(chunk)
            print(f"✅ Downloaded report: {save_path}")
        except Exception as e:
            print(f"❌ Failed to download {report_url} for {reg_no}: {e}")



def start_funcs():
    df = pd.read_csv("cva_data.csv")
    reports = df[["Reg", "Inspection Report"]]
    reg_img = df[["Reg", "Images"]]
    cond_img = df[["Reg", "Damaged_images"]]

    threads = [
        threading.Thread(target=download_reports, args=(reports,)),
        threading.Thread(target=download_images, args=(reg_img, "Images", "Images")),
        threading.Thread(target=download_images, args=(cond_img, "Damaged_images", "Damaged_images")),
    ]

    for t in threads:
        t.start()
    for t in threads:
        t.join()


if __name__ == "__main__":
    start_funcs()


✅ Watermark added to Damaged_images\AE63HWZ\1.jpg
✅ Saved: Damaged_images\AE63HWZ\1.jpg
✅ Watermark added to Images\AE63HWZ\1.jpg
✅ Saved: Images\AE63HWZ\1.jpg
✅ Watermark added to Images\AE63HWZ\2.jpg
✅ Saved: Images\AE63HWZ\2.jpg
✅ Watermark added to Images\AE63HWZ\3.jpg
✅ Saved: Images\AE63HWZ\3.jpg
✅ Watermark added to Images\AE63HWZ\4.jpg
✅ Saved: Images\AE63HWZ\4.jpg
✅ Watermark added to Images\AK66JWJ\1.jpg
✅ Saved: Images\AK66JWJ\1.jpg
✅ Downloaded report: Inspection Report\AE63HWZ\AE63HWZ.pdf
✅ Watermark added to Images\AK66JWJ\2.jpg
✅ Saved: Images\AK66JWJ\2.jpg
✅ Watermark added to Images\AK66JWJ\3.jpg
✅ Saved: Images\AK66JWJ\3.jpg
✅ Watermark added to Images\AK66JWJ\4.jpg
✅ Saved: Images\AK66JWJ\4.jpg
✅ Downloaded report: Inspection Report\AK66JWJ\AK66JWJ.pdf
✅ Watermark added to Images\AO17NFJ\1.jpg
✅ Saved: Images\AO17NFJ\1.jpg
✅ Watermark added to Images\AO17NFJ\2.jpg
✅ Saved: Images\AO17NFJ\2.jpg
✅ Watermark added to Images\AO17NFJ\3.jpg
✅ Saved: Images\AO17NFJ\3.jpg
✅ 

In [4]:
import os
from PyPDF2 import PdfReader, PdfWriter
from reportlab.pdfgen import canvas

HEADER_HEIGHT = 25  
HEADER_TEXT = "Source from Commercial Vehicle Auctions LTD"

def create_header_pdf(page_width, page_height, temp_filename):

    c = canvas.Canvas(temp_filename, pagesize=(page_width, page_height))
 
    c.setFillColorRGB(4/255, 122/255, 250/255)
    c.rect(0, page_height - HEADER_HEIGHT, page_width, HEADER_HEIGHT, fill=1)
    # White text
    c.setFillColorRGB(1, 1, 1)
    c.setFont("Helvetica-Bold", 12)
    text_width = c.stringWidth(HEADER_TEXT, "Helvetica-Bold", 12)
    x = (page_width - text_width) / 2
    y = page_height - HEADER_HEIGHT + 7
    c.drawString(x, y, HEADER_TEXT)
    c.save()

def add_header_to_pdf(input_pdf, output_pdf):

    reader = PdfReader(input_pdf)
    writer = PdfWriter()
    
    for page_number, page in enumerate(reader.pages, start=1):
        page_width = float(page.mediabox.width)
        page_height = float(page.mediabox.height)
        

        temp_header = f"header_temp_{page_number}.pdf"
        create_header_pdf(page_width, page_height, temp_header)
        

        header_reader = PdfReader(temp_header)
        page.merge_page(header_reader.pages[0])
        writer.add_page(page)
        
        os.remove(temp_header)

    with open(output_pdf, "wb") as f:
        writer.write(f)

def add_header_to_all_pdfs(folder):

    for root, dirs, files in os.walk(folder):
        for file in files:
            if file.lower().endswith(".pdf"):
                input_pdf = os.path.join(root, file)
                output_pdf = input_pdf  
                print(f"Adding header to: {input_pdf}")
                add_header_to_pdf(input_pdf, output_pdf)
    print("\n✔ All PDFs in folder and subfolders updated with blue header!")

if __name__ == "__main__":
    folder = "Inspection Report" 
    if not os.path.exists(folder):
        print(f"Folder '{folder}' does not exist!")
    else:
        add_header_to_all_pdfs(folder)


Adding header to: Inspection Report\AE63HWZ\AE63HWZ.pdf
Adding header to: Inspection Report\AK66JWJ\AK66JWJ.pdf
Adding header to: Inspection Report\AO17NFJ\AO17NFJ.pdf
Adding header to: Inspection Report\AP72WAE\AP72WAE.pdf
Adding header to: Inspection Report\AV62AVO\AV62AVO.pdf
Adding header to: Inspection Report\AV66UTH\AV66UTH.pdf
Adding header to: Inspection Report\BD71EKG\BD71EKG.pdf
Adding header to: Inspection Report\BF14XJW\BF14XJW.pdf
Adding header to: Inspection Report\BJ22KHY\BJ22KHY.pdf
Adding header to: Inspection Report\BJ64XBA\BJ64XBA.pdf
Adding header to: Inspection Report\BL16TPU\BL16TPU.pdf
Adding header to: Inspection Report\BL16YBJ\BL16YBJ.pdf
Adding header to: Inspection Report\BN67VWP\BN67VWP.pdf
Adding header to: Inspection Report\BV71ZPU\BV71ZPU.pdf
Adding header to: Inspection Report\BX08RDU\BX08RDU.pdf
Adding header to: Inspection Report\CE56XOM\CE56XOM.pdf
Adding header to: Inspection Report\CF68XHR\CF68XHR.pdf
Adding header to: Inspection Report\CP69BZS\CP69